<a href="https://colab.research.google.com/github/Rasya-ai-web/Machine-Learning-Lab/blob/main/ML_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Random Forest Classifier for Ensemble Learning

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [3]:
df = pd.read_csv("/content/statement_dataset_realistic_200.csv")

print("Dataset loaded successfully!")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nClass Distribution:")
print(df["Label"].value_counts())

Dataset loaded successfully!
   ID                                              Text     Label
0   1                The product has good build quality  Positive
1   2            The quality was better than I expected  Positive
2   3           The instructions were clear and helpful  Positive
3   4       The support team gave an unhelpful response  Negative
4   5  The product design is confusing and inconvenient  Negative

Dataset Shape:
(200, 3)

Class Distribution:
Label
Positive    100
Negative    100
Name: count, dtype: int64


In [4]:
X = df["Text"].astype(str)
y = df["Label"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 160
Testing samples : 40


In [6]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)

TF-IDF training shape: (160, 332)
TF-IDF testing shape : (40, 332)


In [7]:
rf_classifier = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf_classifier,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_tfidf, y_train)

rf_classifier = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(round(grid_search.best_score_, 4))

Fitting 5 folds for each of 72 candidates, totalling 360 fits

Best Parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}

Best Cross-Validation Macro F1:
0.8612


In [8]:
rf_classifier.fit(X_train_tfidf, y_train)

print("Improved Random Forest model trained successfully!")

Improved Random Forest model trained successfully!


In [9]:
y_pred = rf_classifier.predict(X_test_tfidf)

print("Predictions completed!")

Predictions completed!


In [10]:
accuracy = accuracy_score(y_test, y_pred)

print("\nRandom Forest Classification Results")
print("------------------------------------")
print("Accuracy:", round(accuracy, 4))
print("Accuracy Percentage:", round(accuracy * 100, 2), "%")


Random Forest Classification Results
------------------------------------
Accuracy: 0.9
Accuracy Percentage: 90.0 %


In [11]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Negative", "Positive"]
)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[17  3]
 [ 1 19]]


In [12]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=["Negative", "Positive"],
        zero_division=0
    )
)


Classification Report:
              precision    recall  f1-score   support

    Negative       0.94      0.85      0.89        20
    Positive       0.86      0.95      0.90        20

    accuracy                           0.90        40
   macro avg       0.90      0.90      0.90        40
weighted avg       0.90      0.90      0.90        40



In [13]:
print("\nActual Class Distribution:")
print(y_test.value_counts())

print("\nPredicted Class Distribution:")
print(pd.Series(y_pred).value_counts())


Actual Class Distribution:
Label
Negative    20
Positive    20
Name: count, dtype: int64

Predicted Class Distribution:
Positive    22
Negative    18
Name: count, dtype: int64


In [14]:
sample = ["I really enjoyed this product"]

sample_tfidf = vectorizer.transform(sample)

prediction = rf_classifier.predict(sample_tfidf)

print("\nSample Statement:")
print(sample[0])

print("\nPredicted Class:")
print(prediction[0])


Sample Statement:
I really enjoyed this product

Predicted Class:
Positive


In [15]:
sample = ["I really enjoyed this product"]

sample_tfidf = vectorizer.transform(sample)

prediction = rf_classifier.predict(sample_tfidf)

print("\n========================================")
print("       SAMPLE PREDICTION CHECK")
print("========================================")

print("Statement :", sample[0])
print("Prediction :", prediction[0])

if prediction[0] == "Positive":
    print("Result : CORRECT - Positive sentiment detected")
else:
    print("Result : INCORRECT - Model predicted Negative")

print("========================================")


       SAMPLE PREDICTION CHECK
Statement : I really enjoyed this product
Prediction : Positive
Result : CORRECT - Positive sentiment detected
